# LLM-RAG Resolution Service

## Goal

Call the LLM- and RAG-based toponym resolution service for difficult toponym resolution cases.

## What you will do

- Understand the principle of the LLM-RAG method.
- Configure the service endpoint.
- Extract toponyms with one of the NER tools from Notebook 01.
- Choose GeoNames only or GeoNames plus Photon candidates.
- Call `/resolve` and inspect the returned locations.
- Try your own challenging texts and save results.

In [16]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import requests
from src.config import LLM_RAG_BASE_URL, REQUEST_TIMEOUT, RESULTS_DIR
from src.data_utils import load_dataframe_if_exists, save_dataframe
from src.ner_utils import extract_locations_spacy, extract_locations_stanza, extract_locations_flair, extract_locations_transformers, normalize_ner_results

## Step 1: Method idea

This service uses a fine-tuned lightweight LLM together with retrieval-augmented generation. For each text, the service receives already detected toponyms and uses the full context plus candidate locations from geocoders to infer the intended place.

Compared with UniTopRank, this method can be stronger on difficult and challenging cases because the LLM can use richer textual semantics and world knowledge. Typical examples include short ambiguous names, fine-grained places, and cases where local context is needed. The tradeoff is that it needs a running model service and is heavier than the CPU-friendly UniTopRank method.

## Step 2: Configure the service endpoint

Set `SERVICE_BASE_URL` to the running LLM-RAG service. The client calls `POST {SERVICE_BASE_URL}/resolve`.

In [17]:
SERVICE_BASE_URL =  "http://geoparser.intra.dlr.de:8282"

print("SERVICE_BASE_URL:", SERVICE_BASE_URL or "(not set yet)")

SERVICE_BASE_URL: http://geoparser.intra.dlr.de:8282


## Step 3: Extract toponyms from raw text

Use any NER tool introduced in Notebook 01. The default is spaCy because it is lightweight. If a tool or model is missing, go back to Notebook 01 and run that tool's install cell.

In [18]:
text = "My institute locates in Jena, which is in Louisiana, the US."
NER_TOOL = "spacy"  # options: "spacy", "stanza", "flair", "transformers"

def extract_toponyms_with_selected_ner(text, ner_tool="spacy", source_text_id="llm_rag_demo"):
    if ner_tool == "spacy":
        results = extract_locations_spacy(text, model_name="en_core_web_sm")
    elif ner_tool == "stanza":
        results = extract_locations_stanza(text, lang="en")
    elif ner_tool == "flair":
        results = extract_locations_flair(text)
    elif ner_tool == "transformers":
        results = extract_locations_transformers(text)
    else:
        raise ValueError("NER_TOOL must be one of: spacy, stanza, flair, transformers")
    return normalize_ner_results(results, source_text_id=source_text_id)

def mentions_for_service(df, full_text):
    rows = []
    for _, row in df.dropna(subset=["mention"]).iterrows():
        if pd.isna(row.get("start")) or pd.isna(row.get("end")):
            continue
        rows.append({
            "text_id": row.get("source_text_id"),
            "full_text": full_text,
            "mention": str(row["mention"]),
            "start": int(row["start"]),
            "end": int(row["end"]),
        })
    return pd.DataFrame(rows)

ner_rows = extract_toponyms_with_selected_ner(text, NER_TOOL)
mentions = mentions_for_service(ner_rows, text)
mentions

,text_id,full_text,mention,start,end
0,llm_rag_demo,"My institute locates in Jena, which is in Loui...",Jena,24,28
1,llm_rag_demo,"My institute locates in Jena, which is in Loui...",Louisiana,42,51
2,llm_rag_demo,"My institute locates in Jena, which is in Loui...",US,57,59


## Step 4: Choose geocoder support

`use_geonames=True` is required. `use_photon=True` is optional and is useful when Photon/OSM may contain candidates that GeoNames misses, especially fine-grained places.

In [19]:
USE_GEONAMES = True  # required
USE_PHOTON = False    # set True to also use Photon candidates

print("use_geonames:", USE_GEONAMES)
print("use_photon:", USE_PHOTON)

use_geonames: True
use_photon: False


## Step 5: Define the client

The client sends the text, extracted toponyms, and geocoder switches to the `/resolve` endpoint.

In [20]:
def resolve_with_llm_rag(full_text, mention_df, base_url, use_geonames=True, use_photon=False):
    if not base_url:
        print("SERVICE_BASE_URL is not configured. Returning empty result.")
        return []
    if not use_geonames:
        raise ValueError("use_geonames must be True for this service setup.")

    payload = {
        "text": full_text,
        "toponyms": [
            {"text": row["mention"], "start": int(row["start"]), "end": int(row["end"])}
            for _, row in mention_df.iterrows()
        ],
        "use_geonames": True,
        "use_photon": bool(use_photon),
    }

    try:
        response = requests.post(
            f"{base_url.rstrip('/')}/resolve",
            json=payload,
            timeout=max(REQUEST_TIMEOUT, 60),
        )
        response.raise_for_status()
        return response.json().get("results", [])
    except Exception as exc:
        print(f"LLM-RAG service call failed: {exc}")
        return []

## Step 6: Call the service

In [21]:
raw_results = resolve_with_llm_rag(
    full_text=text,
    mention_df=mentions,
    base_url=SERVICE_BASE_URL,
    use_geonames=USE_GEONAMES,
    use_photon=USE_PHOTON,
)
raw_results

[{'text': 'Jena',
  'start': 24,
  'end': 28,
  'address': 'Jena, La Salle Parish, Louisiana, United States',
  'score': -0.0006434519842964928,
  'lat': 31.68323,
  'lon': -92.13374,
  'gid': '4329021',
  'code': 'PPLA2',
  'class': None},
 {'text': 'Louisiana',
  'start': 42,
  'end': 51,
  'address': 'Louisiana, United States',
  'score': -0.0006434519842964928,
  'lat': 31.00047,
  'lon': -92.0004,
  'gid': '4331987',
  'code': 'ADM1',
  'class': None},
 {'text': 'US',
  'start': 57,
  'end': 59,
  'address': 'United States',
  'score': -0.0006434519842964928,
  'lat': 42.05179,
  'lon': -102.33041,
  'gid': '12746282',
  'code': 'RGN',
  'class': None}]

## Step 7: Normalize and save results

The service may not return every optional field. The notebook displays the useful fields and saves the full normalized table for debugging.

In [22]:
def normalize_llm_rag_results(raw_results, service_base_url, use_photon):
    rows = []
    for item in raw_results:
        rows.append({
            "mention": item.get("text") or item.get("mention"),
            "selected_name": item.get("address") or item.get("full_name") or item.get("name"),
            "country": item.get("country") or item.get("country_name"),
            "lat": item.get("lat") or item.get("latitude"),
            "lon": item.get("lon") or item.get("lng") or item.get("longitude"),
            "confidence": item.get("confidence") or item.get("score"),
            "explanation": item.get("explanation") or item.get("reason"),
            "method": "llm_rag_geonames_photon" if use_photon else "llm_rag_geonames",
            "service_base_url": service_base_url,
        })
    return pd.DataFrame(rows)


def compact_llm_rag_results(df):
    display_columns = [
        "mention", "selected_name", "country", "lat", "lon",
        "confidence", "explanation", "method",
    ]
    if df.empty:
        return df[[col for col in display_columns if col in df.columns]]
    cols = []
    for col in display_columns:
        if col not in df.columns:
            continue
        if col in {"mention", "selected_name", "lat", "lon", "method"}:
            cols.append(col)
        elif df[col].notna().any() and (df[col].astype(str).str.len() > 0).any():
            cols.append(col)
    return df[cols]


results = normalize_llm_rag_results(raw_results, SERVICE_BASE_URL, USE_PHOTON)
out = save_dataframe(results, RESULTS_DIR / "llm_rag_results.csv")
print("Saved full result table:", out)
compact_llm_rag_results(results)

Saved full result table: /home/hu_xk/Workplace/wawopensearch3_hackathon/module2_geoparsing/outputs/results/llm_rag_results.csv


,mention,selected_name,lat,lon,confidence,method
0,Jena,"Jena, La Salle Parish, Louisiana, United States",31.68323,-92.13374,-0.000643,llm_rag_geonames
1,Louisiana,"Louisiana, United States",31.00047,-92.00040,-0.000643,llm_rag_geonames
2,US,United States,42.05179,-102.33041,-0.000643,llm_rag_geonames


## Step 8: Play with your own text

Try a difficult case where UniTopRank may struggle. Choose the NER tool, service endpoint, and whether Photon candidates should be used.

In [23]:
#my_text = "The report mentions Victoria Park near London, Ontario"
my_text = "The flood affected Passau and nearby communities along the Danube in Bavaria"
MY_NER_TOOL = "stanza"  # options: "spacy", "stanza", "flair", "transformers"
MY_SERVICE_BASE_URL = SERVICE_BASE_URL
MY_USE_PHOTON = True

my_ner_rows = extract_toponyms_with_selected_ner(
    my_text,
    ner_tool=MY_NER_TOOL,
    source_text_id="my_llm_rag_text",
)
my_mentions = mentions_for_service(my_ner_rows, my_text)

display(my_mentions)

my_raw_results = resolve_with_llm_rag(
    full_text=my_text,
    mention_df=my_mentions,
    base_url=MY_SERVICE_BASE_URL,
    use_geonames=True,
    use_photon=MY_USE_PHOTON,
)

my_results = normalize_llm_rag_results(my_raw_results, MY_SERVICE_BASE_URL, MY_USE_PHOTON)
compact_llm_rag_results(my_results)

,text_id,full_text,mention,start,end
0,my_llm_rag_text,The flood affected Passau and nearby communiti...,Passau,19,25
1,my_llm_rag_text,The flood affected Passau and nearby communiti...,Danube,59,65
2,my_llm_rag_text,The flood affected Passau and nearby communiti...,Bavaria,69,76


,mention,selected_name,lat,lon,confidence,method
0,Passau,"Passau, Lower Bavaria, Germany",48.59944,13.29389,-0.024209,llm_rag_geonames_photon
1,Danube,"Danube River, Germany",48.09585,8.15550,-0.024209,llm_rag_geonames_photon
2,Bavaria,"Bavaria, Germany",49.00000,11.50000,-0.024209,llm_rag_geonames_photon


## Discussion

This service is designed for cases where simple rule-based ranking may not be enough. UniTopRank is fast and interpretable, but it relies on candidate quality and compact ranking signals. The LLM-RAG service can use richer textual context and retrieved geocoder candidates, so it can be stronger on challenging ambiguity. It is also heavier operationally because it requires a running LLM inference service.

## References

- Hu, X., Kersten, J., Klan, F., & Farzana, S. M. (2024). *Toponym resolution leveraging lightweight and open-source large language models and geo-knowledge*. International Journal of Geographical Information Science. https://doi.org/10.1080/13658816.2024.2405182
- Hu, X., Kersten, J., & Klan, F. (2025). *Scalable Toponym Resolution with LLMs: Accuracy and Speed Optimizations*. GeoExT 2025. https://elib.dlr.de/221349/1/paper6.pdf

## Common issues

- `SERVICE_BASE_URL` is empty or points to a service that is not running.
- GeoNames is required; Photon is optional.
- NER may miss a place name before the resolver receives it.
- Some optional fields, such as `country`, `confidence`, or `explanation`, may not be returned by the service. The notebook hides empty optional columns in the display.